In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import pandas as pd
import numpy as np
import json
import torch
import math

In [2]:
file_path = "train.json"
squad_dict = {}
with open(file_path,'r') as f:
  squad_dict = json.load(f)

In [3]:
min_context_len = 10000;
max_context_len = 0;
average_context_size = 0;
total_number_of_context = 0;

In [4]:
questionContext = {};
data = squad_dict['data']
row_Data = []  #creating list
context_index  = 0
cnt = 0
for article in data:
    for paragraph in article['paragraphs']:
        context = paragraph['context']

        context_words = context.split()
        min_context_len = min(min_context_len, len(context_words))
        max_context_len = max(max_context_len, len(context_words))
        total_number_of_context += 1

        for qa in paragraph['qas']:
            if not qa['is_impossible']:
                question = qa['question']
                for ans in qa['answers']:
                    answer = ans['text']
                    start = ans['answer_start']
                    end = start + len(answer)
                    row_Data.append(
                        {
                            'context':context,
                            'question':question,
                            'answer':answer,
                            'start':start,
                            'end':end,
                            'c_id':context_index
                        }
                    )
                    #adding question context
                    questionContext[question] = context_index
        context_index += 1

In [5]:
print("min_context_len : ",min_context_len)
print("max_context_len : ",max_context_len)
print("total_number_of_context : ",total_number_of_context)

counter = 0
for i in questionContext:
  counter+=1;
print("size of question Context : ", counter)

min_context_len :  20
max_context_len :  653
total_number_of_context :  19035
size of question Context :  86769


In [6]:
row = pd.DataFrame(row_Data)

In [7]:
len(questionContext)

l = row['context'].tolist()
documents = row[['context','c_id']].drop_duplicates().reset_index(drop=True)
documents

,context,c_id
0,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,0
1,Following the disbandment of Destiny's Child i...,1
2,"A self-described ""modern-day feminist"", Beyonc...",2
3,"Beyoncé Giselle Knowles was born in Houston, T...",3
4,Beyoncé attended St. Mary's Elementary School ...,4
...,...,...
18877,"Institute of Medicine, the central college of ...",18975
18878,Football and Cricket are the most popular spor...,18976
18879,The total length of roads in Nepal is recorded...,18977
18880,The main international airport serving Kathman...,18978


In [8]:
!pip install transformers faiss-cpu datasets

In [9]:
import faiss
model_name = "facebook/contriever" #Bart  Robert based model # mini llm #
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

/home/a.hunain/anaconda3/envs/tensorflow-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/a.hunain/anaconda3/envs/tensorflow-gpu/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
2025-06-17 16:52:58.650080: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-17 16:52:58.663318: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to r

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [11]:
import torch.nn.functional as F

def encode_text(texts):
    encoded_input = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        output = model(**encoded_input)
        # embeddings = output.last_hidden_state[:, 0]  # CLS token

        #adding new
        embeddings = output.last_hidden_state
        attention_mask = encoded_input['attention_mask'].unsqueeze(-1)
        masked_embeddings = embeddings * attention_mask
        sum_embeddings = masked_embeddings.sum(dim=1)
        lengths = attention_mask.sum(dim=1)
        mean_pooled = sum_embeddings / lengths
        embeddings = F.normalize(mean_pooled, p=2, dim=1)
    return embeddings.cpu().numpy()

def encode_in_batches(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_embeddings = encode_text(batch_texts)
        all_embeddings.append(batch_embeddings)
    return np.vstack(all_embeddings)


In [12]:
texts = documents['context'].tolist()
doc_embeddings = encode_in_batches(texts, batch_size=32)

print("doc_embedding is done !!!")

doc_embedding is done !!!


In [13]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [14]:
question = []
for keys, value in questionContext.items():
  question.append(keys)

In [15]:
import math
import numpy as np

def dcg(scores):
    return sum([rel / math.log2(idx + 2) for idx, rel in enumerate(scores)])

def compute_metrics(question, questionContext, index, k=10, max_samples=2000):
    total_correct = 0
    reciprocal_ranks = []
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    mar_scores = []  # To hold MAR for each query

    for idx, ques in enumerate(question):
        if idx >= max_samples:
            break

        query_embedding = encode_text([ques])
        distances, indices = index.search(query_embedding, k)

        predictions = indices[0]
        actual_index = questionContext[ques]

        # Relevance: 1 if prediction is the correct one
        relevance = [1 if pred == actual_index else 0 for pred in predictions]

        # Accuracy / Recall@k
        if actual_index in predictions:
            total_correct += 1
            rank = list(predictions).index(actual_index)
            reciprocal_ranks.append(1.0 / (rank + 1))
            recall_scores.append(1.0)  # Only one relevant item exists
            precision_scores.append(1.0 / (rank + 1))  # Precision for that one item
        else:
            reciprocal_ranks.append(0.0)
            recall_scores.append(0.0)
            precision_scores.append(0.0)

        # NDCG
        ideal_relevance = sorted(relevance, reverse=True)
        ndcg = dcg(relevance) / dcg(ideal_relevance) if sum(ideal_relevance) > 0 else 0.0
        ndcg_scores.append(ndcg)

        # MAR: For each query, compute recall at k
        num_relevant_items = sum(relevance)  # The number of relevant items in the top-k
        mar_scores.append(num_relevant_items / len(relevance) if num_relevant_items > 0 else 0.0)

    num_samples = min(len(question), max_samples)

    return {
        f"Accuracy@{k}": (total_correct / num_samples) * 100,
        f"MRR@{k}": np.mean(reciprocal_ranks),
        f"NDCG@{k}": np.mean(ndcg_scores),
        f"Recall@{k}": np.mean(recall_scores),
        f"Precision@{k}": np.mean(precision_scores),
        f"MAR@{k}": np.mean(mar_scores)  # Adding MAR to the metrics
    }


In [16]:
samples = [2000, 5000, 10000]    # sample [] random    end center 5000

final_result = []

for spl in samples:
    temp = []
    for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
        results = compute_metrics(
            question = question,
            questionContext=questionContext,
            index = index,
            k=k,
            max_samples=spl
        )
        temp.append(results)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
    final_result.append(temp)


🔍 Top-1 Evaluation Metrics:
Accuracy@1: 32.2000
MRR@1: 0.3220
NDCG@1: 0.3220
Recall@1: 0.3220
Precision@1: 0.3220
MAR@1: 0.3220

🔍 Top-2 Evaluation Metrics:
Accuracy@2: 42.1500
MRR@2: 0.3718
NDCG@2: 0.3848
Recall@2: 0.4215
Precision@2: 0.3718
MAR@2: 0.2107

🔍 Top-3 Evaluation Metrics:
Accuracy@3: 48.2000
MRR@3: 0.3919
NDCG@3: 0.4150
Recall@3: 0.4820
Precision@3: 0.3919
MAR@3: 0.1607

🔍 Top-4 Evaluation Metrics:
Accuracy@4: 51.7500
MRR@4: 0.4008
NDCG@4: 0.4303
Recall@4: 0.5175
Precision@4: 0.4008
MAR@4: 0.1294

🔍 Top-5 Evaluation Metrics:
Accuracy@5: 55.3000
MRR@5: 0.4079
NDCG@5: 0.4440
Recall@5: 0.5530
Precision@5: 0.4079
MAR@5: 0.1106

🔍 Top-10 Evaluation Metrics:
Accuracy@10: 67.6500
MRR@10: 0.4240
NDCG@10: 0.4836
Recall@10: 0.6765
Precision@10: 0.4240
MAR@10: 0.0677

🔍 Top-15 Evaluation Metrics:
Accuracy@15: 74.7000
MRR@15: 0.4296
NDCG@15: 0.5023
Recall@15: 0.7470
Precision@15: 0.4296
MAR@15: 0.0498

🔍 Top-20 Evaluation Metrics:
Accuracy@20: 79.3000
MRR@20: 0.4322
NDCG@20: 0.5132
R

In [17]:
temp = []
for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
        results = compute_metrics(
            question = question,
            questionContext=questionContext,
            index = index,
            k=k,
            max_samples=20000
        )
        temp.append(results)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
final_result.append(temp)


🔍 Top-1 Evaluation Metrics:
Accuracy@1: 13.5900
MRR@1: 0.1359
NDCG@1: 0.1359
Recall@1: 0.1359
Precision@1: 0.1359
MAR@1: 0.1359

🔍 Top-2 Evaluation Metrics:
Accuracy@2: 18.2550
MRR@2: 0.1592
NDCG@2: 0.1653
Recall@2: 0.1825
Precision@2: 0.1592
MAR@2: 0.0913

🔍 Top-3 Evaluation Metrics:
Accuracy@3: 21.0400
MRR@3: 0.1685
NDCG@3: 0.1793
Recall@3: 0.2104
Precision@3: 0.1685
MAR@3: 0.0701

🔍 Top-4 Evaluation Metrics:
Accuracy@4: 22.8900
MRR@4: 0.1731
NDCG@4: 0.1872
Recall@4: 0.2289
Precision@4: 0.1731
MAR@4: 0.0572

🔍 Top-5 Evaluation Metrics:
Accuracy@5: 24.5800
MRR@5: 0.1765
NDCG@5: 0.1938
Recall@5: 0.2458
Precision@5: 0.1765
MAR@5: 0.0492

🔍 Top-10 Evaluation Metrics:
Accuracy@10: 29.8500
MRR@10: 0.1834
NDCG@10: 0.2107
Recall@10: 0.2985
Precision@10: 0.1834
MAR@10: 0.0299

🔍 Top-15 Evaluation Metrics:
Accuracy@15: 33.2350
MRR@15: 0.1861
NDCG@15: 0.2197
Recall@15: 0.3323
Precision@15: 0.1861
MAR@15: 0.0222

🔍 Top-20 Evaluation Metrics:
Accuracy@20: 35.5400
MRR@20: 0.1874
NDCG@20: 0.2251
R

temp = []
for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
        results = compute_metrics(
            question = question,
            questionContext=questionContext,
            index = index,
            k=k,
            max_samples=86769
        )
        temp.append(results)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
final_result.append(temp) 

In [19]:
with open("contriever_final_result.json", "w") as f:
    json.dump(final_result, f)